<a href="https://colab.research.google.com/github/smonodeep/DynamicValuationGovernance/blob/Phase1/000B_NHB_Residex.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
import os
import numpy as np
import pandas as pd

# --- FILE MAPS & PARAMETERS ---
FILE_MAP_2013 = {
    "Bengaluru": "/content/Bengaluru-City HPI Data Current-Q (Base Year 2013).xls",
    "Chennai": "/content/Chennai-City HPI Data Current-Q (Base Year 2013).xls",
    "Delhi": "/content/Delhi-City HPI Data Current-Q (Base Year 2013).xls",
    "Greater Noida": "/content/Greater Noida-City HPI Data Current-Q (Base Year 2013).xls",
    "Hyderabad": "/content/Hyderabad-City HPI Data Current-Q (Base Year 2013).xls",
    "Kolkata": "/content/Kolkata-City HPI Data Current-Q (Base Year 2013).xls",
    "Mumbai": "/content/Mumbai-City HPI Data Current-Q (Base Year 2013).xls",
    "Pune": "/content/Pune-City HPI Data Current-Q (Base Year 2013).xls"
}

FILE_MAP_2018 = {
    "Bengaluru": "/content/Bengaluru-City HPI Data Current-Q (Base Year 2018).xls",
    "Chennai": "/content/Chennai-City HPI Data Current-Q (Base Year 2018).xls",
    "Delhi": "/content/Delhi-City HPI Data Current-Q (Base Year 2018).xls",
    "Greater Noida": "/content/Greater Noida-City HPI Data Current-Q (Base Year 2018).xls",
    "Hyderabad": "/content/Hyderabad-City HPI Data Current-Q (Base Year 2018).xls",
    "Kolkata": "/content/Kolkata-City HPI Data Current-Q (Base Year 2018).xls",
    "Mumbai": "/content/Mumbai-City HPI Data Previous-Q (Base Year 2018).xls",
    "Pune": "/content/Pune-City HPI Data Previous-Q (Base Year 2018).xls"
}

LINKING_FACTORS = {
    "Bengaluru": 1.36,
    "Chennai": 1.28,
    "Delhi": 1.01,
    "Greater Noida": 1.23,
    "Hyderabad": 1.32,
    "Kolkata": 1.29,
    "Mumbai": 1.43,
    "Pune": 1.46
}

def inspect_and_read_xls(filepath):
    """Checks if a file is HTML disguised as XLS or genuine XLS, then parses correctly."""
    with open(filepath, 'rb') as f:
        start_bytes = f.read(100)

    is_html = b'<html' in start_bytes.lower() or b'<table' in start_bytes.lower()

    if is_html:
        tables = pd.read_html(filepath)
        for df in tables:
            df_str = df.astype(str).values
            if any('quarter' in val.lower() or 'period' in val.lower() for val in df_str.flatten()):
                return df
        return max(tables, key=len)
    else:
        xls = pd.ExcelFile(filepath)
        for sheet in xls.sheet_names:
            df = xls.parse(sheet)
            df_str = df.astype(str).values
            if any('quarter' in val.lower() or 'period' in val.lower() for val in df_str.flatten()):
                return df
        return xls.parse(xls.sheet_names[0])

def clean_quarter_string(q_str):
    """Standardizes quarterly strings to the format 'Month-Year' (e.g., Jun-2013)."""
    if not isinstance(q_str, str):
        return None

    s = q_str.strip().replace(' ', '').replace('"', '').replace('\'', '')
    parts = s.split('-')
    if len(parts) >= 1:
        month_part = parts[0][:3].strip().capitalize()
        year_part = ""
        if len(parts) >= 2:
            year_part = ''.join(filter(str.isdigit, parts[1]))
        else:
            year_part = ''.join(filter(str.isdigit, s))

        if len(year_part) >= 4:
            year = year_part[:4]
        elif len(year_part) == 2:
            year = "20" + year_part
        else:
            match_digits = [int(i) for i in s.split() if i.isdigit()]
            if match_digits:
                year = str(match_digits[0])
                if len(year) == 2:
                    year = "20" + year
            else:
                return None

        valid_months = ['Jun', 'Sep', 'Dec', 'Mar']
        if month_part in valid_months:
            return f"{month_part}-{year}"

    return None

def parse_hpi_data(df, city, base_year, filename):
    """Robustly extracts the quarter index values and labels."""
    df = df.copy()

    if isinstance(df.columns, pd.MultiIndex):
        df.columns = ['_'.join([str(c) for c in col]).strip() for col in df.columns.values]
    else:
        df.columns = [str(c).strip() for c in df.columns]

    quarter_col = None
    for col in df.columns:
        if any(x in col.lower() for x in ['quarter', 'period', 'date', 'year']):
            quarter_col = col
            break

    index_col = None
    target_keys = ['hpi@assessment prices', 'composite index', 'composite', 'hpi @ assessment prices', 'hpi']
    for key in target_keys:
        for col in df.columns:
            if key in col.lower() and not any(x in col.lower() for x in ['usable', 'record', 'carpet', 'percentage', 'share', 'pct']):
                index_col = col
                break
        if index_col:
            break

    if quarter_col is None or index_col is None:
        for i in range(min(10, len(df))):
            row_vals = [str(v).lower() for v in df.iloc[i].values]
            for idx, val in enumerate(row_vals):
                if 'quarter' in val or 'period' in val:
                    quarter_col = df.columns[idx]
                if 'hpi' in val or 'composite' in val:
                    index_col = df.columns[idx]

    if quarter_col is None:
        quarter_col = df.columns[0]
    if index_col is None:
        index_col = df.columns[1]

    cleaned_rows = []
    for idx, row in df.iterrows():
        raw_q = str(row[quarter_col])
        clean_q = clean_quarter_string(raw_q)
        if clean_q:
            try:
                val = float(str(row[index_col]).replace(',', '').strip())
                if not np.isnan(val):
                    cleaned_rows.append({
                        'City': city,
                        'Quarter': clean_q,
                        'HPI_Raw': val,
                        'Source_File': os.path.basename(filename)
                    })
            except Exception:
                continue

    return pd.DataFrame(cleaned_rows)

# --- PROCESS FILES ---
list_table_a = []
list_table_b = []

for city, filepath in FILE_MAP_2013.items():
    df_raw = inspect_and_read_xls(filepath)
    df_extracted = parse_hpi_data(df_raw, city, 2013, filepath)
    valid_quarters = [
        "Jun-2013", "Sep-2013", "Dec-2013", "Mar-2014",
        "Jun-2014", "Sep-2014", "Dec-2014", "Mar-2015",
        "Jun-2015", "Sep-2015", "Dec-2015", "Mar-2016",
        "Jun-2016", "Sep-2016", "Dec-2016", "Mar-2017",
        "Jun-2017", "Sep-2017", "Dec-2017", "Mar-2018"
    ]
    df_extracted = df_extracted[df_extracted['Quarter'].isin(valid_quarters)]
    list_table_a.append(df_extracted)

for city, filepath in FILE_MAP_2018.items():
    df_raw = inspect_and_read_xls(filepath)
    df_extracted = parse_hpi_data(df_raw, city, 2018, filepath)
    valid_quarters = [
        "Jun-2018", "Sep-2018", "Dec-2018", "Mar-2019",
        "Jun-2019", "Sep-2019", "Dec-2019", "Mar-2020",
        "Jun-2020", "Sep-2020", "Dec-2020", "Mar-2021",
        "Jun-2021", "Sep-2021", "Dec-2021", "Mar-2022",
        "Jun-2022", "Sep-2022", "Dec-2022", "Mar-2023",
        "Jun-2023", "Sep-2023", "Dec-2023", "Mar-2024",
        "Jun-2024", "Sep-2024", "Dec-2024", "Mar-2025",
        "Jun-2025", "Sep-2025", "Dec-2025"
    ]
    df_extracted = df_extracted[df_extracted['Quarter'].isin(valid_quarters)]
    list_table_b.append(df_extracted)

# --- SORT HELPER ---
quarter_order = [
    "Jun-2013", "Sep-2013", "Dec-2013", "Mar-2014",
    "Jun-2014", "Sep-2014", "Dec-2014", "Mar-2015",
    "Jun-2015", "Sep-2015", "Dec-2015", "Mar-2016",
    "Jun-2016", "Sep-2016", "Dec-2016", "Mar-2017",
    "Jun-2017", "Sep-2017", "Dec-2017", "Mar-2018",
    "Jun-2018", "Sep-2018", "Dec-2018", "Mar-2019",
    "Jun-2019", "Sep-2019", "Dec-2019", "Mar-2020",
    "Jun-2020", "Sep-2020", "Dec-2020", "Mar-2021",
    "Jun-2021", "Sep-2021", "Dec-2021", "Mar-2022",
    "Jun-2022", "Sep-2022", "Dec-2022", "Mar-2023",
    "Jun-2023", "Sep-2023", "Dec-2023", "Mar-2024",
    "Jun-2024", "Sep-2024", "Dec-2024", "Mar-2025",
    "Jun-2025", "Sep-2025", "Dec-2025"
]
q_order_map = {q: i for i, q in enumerate(quarter_order)}

def sort_df_by_city_and_quarter(df):
    df = df.copy()
    df['q_sort'] = df['Quarter'].map(q_order_map)
    df = df.sort_values(by=['City', 'q_sort']).drop(columns=['q_sort'])
    return df.reset_index(drop=True)

# --- STEP 2: TABLE A (RAW 2013-BASE LONG & WIDE) ---
TABLE_A_2013_RAW = pd.concat(list_table_a, ignore_index=True)
TABLE_A_2013_RAW = TABLE_A_2013_RAW.rename(columns={'HPI_Raw': 'HPI_2013_Base'})
TABLE_A_2013_RAW = sort_df_by_city_and_quarter(TABLE_A_2013_RAW)

TABLE_A_2013_WIDE = TABLE_A_2013_RAW.pivot(index='Quarter', columns='City', values='HPI_2013_Base')
TABLE_A_2013_WIDE = TABLE_A_2013_WIDE.reindex(quarter_order[:20])
TABLE_A_2013_WIDE.index.name = 'Period'
TABLE_A_2013_WIDE = TABLE_A_2013_WIDE.reset_index()

# --- STEP 3: TABLE B (RAW 2018-BASE LONG & WIDE) ---
TABLE_B_2018_RAW = pd.concat(list_table_b, ignore_index=True)
TABLE_B_2018_RAW = TABLE_B_2018_RAW.rename(columns={'HPI_Raw': 'HPI_2018_Base_Raw'})
TABLE_B_2018_RAW = sort_df_by_city_and_quarter(TABLE_B_2018_RAW)

TABLE_B_2018_WIDE = TABLE_B_2018_RAW.pivot(index='Quarter', columns='City', values='HPI_2018_Base_Raw')
TABLE_B_2018_WIDE = TABLE_B_2018_WIDE.reindex(quarter_order[20:])
TABLE_B_2018_WIDE.index.name = 'Period'
TABLE_B_2018_WIDE = TABLE_B_2018_WIDE.reset_index()

# --- STEP 4: NHB LINKING FACTORS ---
linking_factors_data = [{'City': k, 'Forward_Linking_Factor': v} for k, v in LINKING_FACTORS.items()]
TABLE_LINKING_FACTORS = pd.DataFrame(linking_factors_data).sort_values('City').reset_index(drop=True)

# --- STEP 5: VALIDATE LINKING FACTORS ---
validation_results = []
for city in LINKING_FACTORS.keys():
    city_df = TABLE_A_2013_RAW[TABLE_A_2013_RAW['City'] == city]
    transition_quarters = ["Jun-2017", "Sep-2017", "Dec-2017", "Mar-2018"]
    fy1718_df = city_df[city_df['Quarter'].isin(transition_quarters)]

    original_mean = fy1718_df['HPI_2013_Base'].mean()
    linking_factor = LINKING_FACTORS[city]
    converted_mean = (fy1718_df['HPI_2013_Base'] / linking_factor).mean()
    difference_100 = converted_mean - 100.0

    validation_results.append({
        'City': city,
        'Original_FY2017_18_Mean': original_mean,
        'Linking_Factor': linking_factor,
        'Converted_FY2017_18_Mean': converted_mean,
        'Difference_From_100': difference_100
    })
TABLE_VALIDATION = pd.DataFrame(validation_results).sort_values('City').reset_index(drop=True)

# --- STEP 6: CREATE TABLE C: HARMONIZED LONG ---
list_harmonized = []

for city in LINKING_FACTORS.keys():
    city_df = TABLE_A_2013_RAW[TABLE_A_2013_RAW['City'] == city].copy()
    lf = LINKING_FACTORS[city]
    city_df['Base_Source'] = "2013"
    city_df['HPI_Original'] = city_df['HPI_2013_Base']
    city_df['Linking_Factor'] = lf
    city_df['HPI_2018_Base'] = city_df['HPI_Original'] / lf
    list_harmonized.append(city_df[['City', 'Quarter', 'Base_Source', 'HPI_Original', 'Linking_Factor', 'HPI_2018_Base']])

for city in LINKING_FACTORS.keys():
    city_df = TABLE_B_2018_RAW[TABLE_B_2018_RAW['City'] == city].copy()
    city_df['Base_Source'] = "2018"
    city_df['HPI_Original'] = city_df['HPI_2018_Base_Raw']
    city_df['Linking_Factor'] = 1.0
    city_df['HPI_2018_Base'] = city_df['HPI_Original']
    list_harmonized.append(city_df[['City', 'Quarter', 'Base_Source', 'HPI_Original', 'Linking_Factor', 'HPI_2018_Base']])

TABLE_C_HARMONIZED = pd.concat(list_harmonized, ignore_index=True)
TABLE_C_HARMONIZED['Period'] = TABLE_C_HARMONIZED['Quarter']
TABLE_C_HARMONIZED = sort_df_by_city_and_quarter(TABLE_C_HARMONIZED)
TABLE_C_HARMONIZED = TABLE_C_HARMONIZED[['City', 'Quarter', 'Period', 'Base_Source', 'HPI_Original', 'Linking_Factor', 'HPI_2018_Base']]

# --- STEP 7: DERIVED GROWTH VARIABLES ---
TABLE_C_WITH_GROWTH = TABLE_C_HARMONIZED.copy()
TABLE_C_WITH_GROWTH['q_sort'] = TABLE_C_WITH_GROWTH['Quarter'].map(q_order_map)
TABLE_C_WITH_GROWTH = TABLE_C_WITH_GROWTH.sort_values(by=['City', 'q_sort']).reset_index(drop=True)
TABLE_C_WITH_GROWTH['log_HPI'] = np.log(TABLE_C_WITH_GROWTH['HPI_2018_Base'])

list_with_growth = []
for city, group in TABLE_C_WITH_GROWTH.groupby('City', sort=False):
    group = group.copy()
    group['QoQ_HPI_Growth'] = 100 * (group['log_HPI'] - group['log_HPI'].shift(1))
    group['YoY_HPI_Growth'] = 100 * (group['log_HPI'] - group['log_HPI'].shift(4))
    list_with_growth.append(group)

TABLE_C_WITH_GROWTH = pd.concat(list_with_growth, ignore_index=True)
TABLE_C_WITH_GROWTH = TABLE_C_WITH_GROWTH.drop(columns=['q_sort'])

# --- STEP 8: RECONCILIATION TESTS ---
checks = {}
checks['CHECK 1: TABLE A (160 obs)'] = (len(TABLE_A_2013_RAW) == 160)
checks['CHECK 2: TABLE B (248 obs)'] = (len(TABLE_B_2018_RAW) == 248)
checks['CHECK 3: TABLE C (408 obs)'] = (len(TABLE_C_HARMONIZED) == 408)
checks['CHECK 4: 8 Unique Cities'] = (TABLE_C_HARMONIZED['City'].nunique() == 8)

qc_city = TABLE_C_HARMONIZED.groupby('City')['Quarter'].count()
checks['CHECK 5: 51 Quarters per City'] = all(qc_city == 51)
checks['CHECK 6: No duplicate City + Quarter'] = not TABLE_C_HARMONIZED.duplicated(subset=['City', 'Quarter']).any()
checks['CHECK 7: No missing HPI values'] = not TABLE_C_HARMONIZED['HPI_2018_Base'].isnull().any()

hist_df = TABLE_C_HARMONIZED[TABLE_C_HARMONIZED['Base_Source'] == "2013"]
checks['CHECK 8: Historical period ends Mar-2018'] = (hist_df['Period'].unique()[-1] == 'Mar-2018')

curr_df = TABLE_C_HARMONIZED[TABLE_C_HARMONIZED['Base_Source'] == "2018"]
checks['CHECK 9: 2018-base period starts Jun-2018'] = (curr_df['Period'].unique()[0] == 'Jun-2018')
checks['CHECK 10: No duplicated transition quarter'] = (len(set(hist_df['Period'].unique()).intersection(set(curr_df['Period'].unique()))) == 0)

factors_correct = True
for city, factor in LINKING_FACTORS.items():
    subset = TABLE_C_HARMONIZED[(TABLE_C_HARMONIZED['City'] == city) & (TABLE_C_HARMONIZED['Base_Source'] == '2013')]
    if not (subset['Linking_Factor'] == factor).all():
        factors_correct = False
checks['CHECK 11: All linking factors match specified values'] = factors_correct
checks['CHECK 12: FY17-18 converted means approx 100'] = all(abs(TABLE_VALIDATION['Converted_FY2017_18_Mean'] - 100.0) < 5)

checks['CHECK 13: 2018 observations unchanged'] = np.allclose(
    TABLE_C_HARMONIZED[TABLE_C_HARMONIZED['Base_Source'] == '2018']['HPI_2018_Base'],
    TABLE_C_HARMONIZED[TABLE_C_HARMONIZED['Base_Source'] == '2018']['HPI_Original']
)

hist_subset = TABLE_C_HARMONIZED[TABLE_C_HARMONIZED['Base_Source'] == '2013']
checks['CHECK 14: Historical observations equal HPI_Original / Factor'] = np.allclose(
    hist_subset['HPI_2018_Base'],
    hist_subset['HPI_Original'] / hist_subset['Linking_Factor']
)

print("Reconciliation Checks Status:")
for k, v in checks.items():
    print(f"{k}: {'PASSED' if v else 'FAILED'}")

# --- STEP 9: RECONSTRUCT WIDE TABLES ---
TABLE_C_WIDE = TABLE_C_HARMONIZED.pivot(index='Period', columns='City', values='HPI_2018_Base')
TABLE_C_WIDE = TABLE_C_WIDE.reindex(quarter_order)
TABLE_C_WIDE = TABLE_C_WIDE.reset_index()

# --- STEP 10: EXPORT TO FILES ---
TABLE_A_2013_RAW.to_csv("NHB_RESIDEX_2013_RAW_LONG.csv", index=False)
TABLE_A_2013_WIDE.to_csv("NHB_RESIDEX_2013_RAW_WIDE.csv", index=False)
TABLE_B_2018_RAW.to_csv("NHB_RESIDEX_2018_RAW_LONG.csv", index=False)
TABLE_B_2018_WIDE.to_csv("NHB_RESIDEX_2018_RAW_WIDE.csv", index=False)
TABLE_LINKING_FACTORS.to_csv("NHB_RESIDEX_LINKING_FACTORS.csv", index=False)
TABLE_C_HARMONIZED.to_csv("NHB_RESIDEX_FULL_HARMONIZED_LONG.csv", index=False)
TABLE_C_WIDE.to_csv("NHB_RESIDEX_FULL_HARMONIZED_WIDE.csv", index=False)
TABLE_C_WITH_GROWTH.to_csv("NHB_RESIDEX_FULL_HARMONIZED_WITH_GROWTH.csv", index=False)

# --- STEP 11: FINAL AUDIT REPORT ---
print("\n" + "=" * 60)
print("                   FINAL AUDIT REPORT                     ")
print("=" * 60)
print("RAW 2013 DATA")
print(f"Rows:             {len(TABLE_A_2013_RAW)}")
print(f"Cities:           {sorted(list(TABLE_A_2013_RAW['City'].unique()))}")
print(f"Start:            {TABLE_A_2013_RAW['Quarter'].iloc[0]}")
print(f"End:              {TABLE_A_2013_RAW['Quarter'].iloc[-1]}")
print("-" * 40)
print("RAW 2018 DATA")
print(f"Rows:             {len(TABLE_B_2018_RAW)}")
print(f"Cities:           {sorted(list(TABLE_B_2018_RAW['City'].unique()))}")
print(f"Start:            {TABLE_B_2018_RAW['Quarter'].iloc[0]}")
print(f"End:              {TABLE_B_2018_RAW['Quarter'].iloc[-1]}")
print("-" * 40)
print("HARMONIZED DATA")
print(f"Rows:             {len(TABLE_C_HARMONIZED)}")
print(f"Cities:           {len(TABLE_C_HARMONIZED['City'].unique())}")
print(f"Quarters per city: {len(TABLE_C_HARMONIZED) // len(TABLE_C_HARMONIZED['City'].unique())}")
print(f"Start:            {quarter_order[0]}")
print(f"End:              {quarter_order[-1]}")
print("-" * 40)
print("LINKING VALIDATION")
print(TABLE_VALIDATION.to_string(index=False, formatters={
    'Original_FY2017_18_Mean': '{:,.2f}'.format,
    'Linking_Factor': '{:,.2f}'.format,
    'Converted_FY2017_18_Mean': '{:,.2f}'.format,
    'Difference_From_100': '{:,.4f}'.format
}))
print("-" * 40)
print("DATA QUALITY")
print(f"Duplicate city-quarter rows: {TABLE_C_HARMONIZED.duplicated(subset=['City', 'Quarter']).sum()}")
print(f"Missing HPI:                 {TABLE_C_HARMONIZED['HPI_2018_Base'].isnull().sum()}")

missing_quarters_by_city = []
for city, group in TABLE_C_HARMONIZED.groupby('City'):
    missing_cnt = len(set(quarter_order) - set(group['Quarter']))
    missing_quarters_by_city.append(f"{city}: {missing_cnt}")
print(f"Missing quarters:            {', '.join(missing_quarters_by_city)}")

unexpected_cities = set(TABLE_C_HARMONIZED['City'].unique()) - set(LINKING_FACTORS.keys())
print(f"Unexpected cities:           {list(unexpected_cities) if unexpected_cities else 'None'}")

unexpected_dates = set(TABLE_C_HARMONIZED['Quarter'].unique()) - set(quarter_order)
print(f"Unexpected dates:            {list(unexpected_dates) if unexpected_dates else 'None'}")
print("=" * 60)
print("               DEMONSTRATION OUTPUTS                      ")
print("=" * 60)

print("\n1. First 10 rows of TABLE A (RAW 2013 LONG):")
print(TABLE_A_2013_RAW.head(10).to_string(index=False))

print("\n2. First 10 rows of TABLE B (RAW 2018 LONG):")
print(TABLE_B_2018_RAW.head(10).to_string(index=False))

print("\n3. First 10 rows of TABLE C (HARMONIZED LONG):")
print(TABLE_C_HARMONIZED.head(10).to_string(index=False))

print("\n4. Last 10 rows of TABLE C (HARMONIZED LONG):")
print(TABLE_C_HARMONIZED.tail(10).to_string(index=False))

print("\n5. Linking Validation Table:")
print(TABLE_VALIDATION.to_string(index=False))

print("\n6. Complete Wide Harmonized HPI Dataset (51 Quarters x 8 Cities):")
print(TABLE_C_WIDE.to_string(index=False))


Reconciliation Checks Status:
CHECK 1: TABLE A (160 obs): PASSED
CHECK 2: TABLE B (248 obs): PASSED
CHECK 3: TABLE C (408 obs): PASSED
CHECK 4: 8 Unique Cities: PASSED
CHECK 5: 51 Quarters per City: PASSED
CHECK 6: No duplicate City + Quarter: PASSED
CHECK 7: No missing HPI values: PASSED
CHECK 8: Historical period ends Mar-2018: PASSED
CHECK 9: 2018-base period starts Jun-2018: PASSED
CHECK 10: No duplicated transition quarter: PASSED
CHECK 11: All linking factors match specified values: PASSED
CHECK 12: FY17-18 converted means approx 100: PASSED
CHECK 13: 2018 observations unchanged: PASSED
CHECK 14: Historical observations equal HPI_Original / Factor: PASSED

                   FINAL AUDIT REPORT                     
RAW 2013 DATA
Rows:             160
Cities:           ['Bengaluru', 'Chennai', 'Delhi', 'Greater Noida', 'Hyderabad', 'Kolkata', 'Mumbai', 'Pune']
Start:            Jun-2013
End:              Mar-2018
----------------------------------------
RAW 2018 DATA
Rows:         

Further Validate this data created manually

In [7]:
import pandas as pd
import numpy as np

# Map our string quarters ('Jun-2013') to match standard macroeconomic periods ('2013Q2')
quarter_to_macro = {
    "Jun-2013": "2013Q2", "Sep-2013": "2013Q3", "Dec-2013": "2013Q4", "Mar-2014": "2014Q1",
    "Jun-2014": "2014Q2", "Sep-2014": "2014Q3", "Dec-2014": "2014Q4", "Mar-2015": "2015Q1",
    "Jun-2015": "2015Q2", "Sep-2015": "2015Q3", "Dec-2015": "2015Q4", "Mar-2016": "2016Q1",
    "Jun-2016": "2016Q2", "Sep-2016": "2016Q3", "Dec-2016": "2016Q4", "Mar-2017": "2017Q1",
    "Jun-2017": "2017Q2", "Sep-2017": "2017Q3", "Dec-2017": "2017Q4", "Mar-2018": "2018Q1",
    "Jun-2018": "2018Q2", "Sep-2018": "2018Q3", "Dec-2018": "2018Q4", "Mar-2019": "2019Q1",
    "Jun-2019": "2019Q2", "Sep-2019": "2019Q3", "Dec-2019": "2019Q4", "Mar-2020": "2020Q1",
    "Jun-2020": "2020Q2", "Sep-2020": "2020Q3", "Dec-2020": "2020Q4", "Mar-2021": "2021Q1",
    "Jun-2021": "2021Q2", "Sep-2021": "2021Q3", "Dec-2021": "2021Q4", "Mar-2022": "2022Q1",
    "Jun-2022": "2022Q2", "Sep-2022": "2022Q3", "Dec-2022": "2022Q4", "Mar-2023": "2023Q1",
    "Jun-2023": "2023Q2", "Sep-2023": "2023Q3", "Dec-2023": "2023Q4", "Mar-2024": "2024Q1",
    "Jun-2024": "2024Q2", "Sep-2024": "2024Q3", "Dec-2024": "2024Q4", "Mar-2025": "2025Q1",
    "Jun-2025": "2025Q2", "Sep-2025": "2025Q3", "Dec-2025": "2025Q4"
}

# Define paths
manual_long_path = "/content/NHB_RESIDEX_8City_FULL_HARMONIZED_2013_2025_m.csv"
manual_wide_path = "/content/NHB_RESIDEX_8City_FULL_HARMONIZED_WIDE_2013_2025_m.csv"

df_manual_long = pd.read_csv(manual_long_path)
df_manual_wide = pd.read_csv(manual_wide_path)

print("=== LOADING & RE-ALIGNING TIMELINES ===")

# Format dynamic table to use macro periods
df_dynamic_wide_sorted = TABLE_C_WIDE.copy()
df_dynamic_wide_sorted['Period'] = df_dynamic_wide_sorted['Period'].map(quarter_to_macro)
df_dynamic_wide_sorted = df_dynamic_wide_sorted.sort_values('Period').reset_index(drop=True)

# Format manual table
df_manual_wide_sorted = df_manual_wide.sort_values('Period').reset_index(drop=True)

# Compare matching columns across chronological index order
wide_cols = sorted([col for col in TABLE_C_WIDE.columns if col != 'Period'])

diff_wide = np.abs(df_manual_wide_sorted[wide_cols].values - df_dynamic_wide_sorted[wide_cols].values)
max_diff_wide = np.nanmax(diff_wide)

print(f"Corrected Maximum difference (Chronologically Aligned): {max_diff_wide:.10f}")
if max_diff_wide < 1e-4:
    print("SUCCESS: The manually created files and our pipeline outputs are in PERFECT mathematical parity! 🎉")
else:
    print("WARNING: Unaligned numbers found even after chronological mapping.")

=== LOADING & RE-ALIGNING TIMELINES ===
Corrected Maximum difference (Chronologically Aligned): 0.0000000000
SUCCESS: The manually created files and our pipeline outputs are in PERFECT mathematical parity! 🎉


In [8]:
import pandas as pd
import numpy as np

# Define paths for the files to validate
manual_long_path = "/content/NHB_RESIDEX_8City_FULL_HARMONIZED_2013_2025_m.csv"
manual_wide_path = "/content/NHB_RESIDEX_8City_FULL_HARMONIZED_WIDE_2013_2025_m.csv"

pipeline_long_path = "/content/NHB_RESIDEX_FULL_HARMONIZED_WITH_GROWTH.csv"
pipeline_wide_path = "/content/NHB_RESIDEX_FULL_HARMONIZED_WIDE.csv"

# Load datasets
df_man_long = pd.read_csv(manual_long_path)
df_man_wide = pd.read_csv(manual_wide_path)
df_pipe_long = pd.read_csv(pipeline_long_path)
df_pipe_wide = pd.read_csv(pipeline_wide_path)

print("============================================================")
print("             CROSS-DATASET VALIDATION REPORT                ")
print("============================================================\n")

# 1. Structural Parity Checks
print("--- SECTION 1: SHAPE & STRUCTURE MATCHING ---")
print(f"Manual Long shape:   {df_man_long.shape} | Pipeline Long shape:   {df_pipe_long.shape}")
print(f"Manual Wide shape:   {df_man_wide.shape}  | Pipeline Wide shape:   {df_pipe_wide.shape}")

shape_long_ok = (df_man_long.shape[0] == df_pipe_long.shape[0])
shape_wide_ok = (df_man_wide.shape[0] == df_pipe_wide.shape[0])
print(f"Long Shape Match:    {'PASSED' if shape_long_ok else 'FAILED'}")
print(f"Wide Shape Match:    {'PASSED' if shape_wide_ok else 'FAILED'}\n")

# 2. Chronological Level Parity
print("--- SECTION 2: INDEX LEVEL COMPARISON ---")
# Map pipeline quarters to standard macroeconomic formats to align periods
quarter_to_macro = {
    "Jun-2013": "2013Q2", "Sep-2013": "2013Q3", "Dec-2013": "2013Q4", "Mar-2014": "2014Q1",
    "Jun-2014": "2014Q2", "Sep-2014": "2014Q3", "Dec-2014": "2014Q4", "Mar-2015": "2015Q1",
    "Jun-2015": "2015Q2", "Sep-2015": "2015Q3", "Dec-2015": "2015Q4", "Mar-2016": "2016Q1",
    "Jun-2016": "2016Q2", "Sep-2016": "2016Q3", "Dec-2016": "2016Q4", "Mar-2017": "2017Q1",
    "Jun-2017": "2017Q2", "Sep-2017": "2017Q3", "Dec-2017": "2017Q4", "Mar-2018": "2018Q1",
    "Jun-2018": "2018Q2", "Sep-2018": "2018Q3", "Dec-2018": "2018Q4", "Mar-2019": "2019Q1",
    "Jun-2019": "2019Q2", "Sep-2019": "2019Q3", "Dec-2019": "2019Q4", "Mar-2020": "2020Q1",
    "Jun-2020": "2020Q2", "Sep-2020": "2020Q3", "Dec-2020": "2020Q4", "Mar-2021": "2021Q1",
    "Jun-2021": "2021Q2", "Sep-2021": "2021Q3", "Dec-2021": "2021Q4", "Mar-2022": "2022Q1",
    "Jun-2022": "2022Q2", "Sep-2022": "2022Q3", "Dec-2022": "2022Q4", "Mar-2023": "2023Q1",
    "Jun-2023": "2023Q2", "Sep-2023": "2023Q3", "Dec-2023": "2023Q4", "Mar-2024": "2024Q1",
    "Jun-2024": "2024Q2", "Sep-2024": "2024Q3", "Dec-2024": "2024Q4", "Mar-2025": "2025Q1",
    "Jun-2025": "2025Q2", "Sep-2025": "2025Q3", "Dec-2025": "2025Q4"
}

df_pipe_wide_aligned = df_pipe_wide.copy()
df_pipe_wide_aligned['Period'] = df_pipe_wide_aligned['Period'].map(quarter_to_macro)
df_pipe_wide_aligned = df_pipe_wide_aligned.sort_values('Period').reset_index(drop=True)
df_man_wide_sorted = df_man_wide.sort_values('Period').reset_index(drop=True)

cities = sorted([col for col in df_pipe_wide.columns if col != 'Period'])

max_index_diff = 0.0
discrepancies = []

for city in cities:
    diffs = np.abs(df_pipe_wide_aligned[city].values - df_man_wide_sorted[city].values)
    max_diff = np.max(diffs)
    if max_diff > max_index_diff:
        max_index_diff = max_diff
    if max_diff > 1e-5:
        discrepancies.append((city, max_diff))

print(f"Maximum numerical index value discrepancy: {max_index_diff:.10f}")
if not discrepancies:
    print("HPI Index Levels:   PASSED (100% Identical Across All Cities & Quarters)")
else:
    print("HPI Index Levels:   FAILED (Discrepancies found)")
    for city, diff in discrepancies:
        print(f"  - {city}: max diff {diff:.6f}")

# 3. Growth Metric Verification
print("\n--- SECTION 3: QoQ & YoY LOG-GROWTH RATE PARITY ---")
try:
    # Align Long Datasets
    df_pipe_long_aligned = df_pipe_long.copy()
    df_pipe_long_aligned['Period'] = df_pipe_long_aligned['Quarter'].map(quarter_to_macro)
    df_pipe_long_aligned = df_pipe_long_aligned.sort_values(['City', 'Period']).reset_index(drop=True)

    df_man_long_aligned = df_man_long.copy()
    df_man_long_aligned = df_man_long_aligned.sort_values(['City', 'Period']).reset_index(drop=True)

    # Compare values ignoring initial NaN rows
    mask = df_pipe_long_aligned['QoQ_HPI_Growth'].notnull() & df_man_long_aligned['HPI_growth_qoq_pct'].notnull()
    max_qoq_diff = np.max(np.abs(df_pipe_long_aligned.loc[mask, 'QoQ_HPI_Growth'].values - df_man_long_aligned.loc[mask, 'HPI_growth_qoq_pct'].values))

    mask_yoy = df_pipe_long_aligned['YoY_HPI_Growth'].notnull() & df_man_long_aligned['HPI_growth_yoy_pct'].notnull()
    max_yoy_diff = np.max(np.abs(df_pipe_long_aligned.loc[mask_yoy, 'YoY_HPI_Growth'].values - df_man_long_aligned.loc[mask_yoy, 'HPI_growth_yoy_pct'].values))

    print(f"Max QoQ Growth rate difference: {max_qoq_diff:.10f}")
    print(f"Max YoY Growth rate difference: {max_yoy_diff:.10f}")

    if max_qoq_diff < 1e-4 and max_yoy_diff < 1e-4:
        print("Growth Metrics check: PASSED (Log rates match perfectly)")
    else:
        print("Growth Metrics check: WARNING (Small numerical calculation variations found)")
except Exception as e:
    print(f"Growth Parity Check failed: {e}")

print("\n============================================================")


             CROSS-DATASET VALIDATION REPORT                

--- SECTION 1: SHAPE & STRUCTURE MATCHING ---
Manual Long shape:   (408, 9) | Pipeline Long shape:   (408, 10)
Manual Wide shape:   (51, 9)  | Pipeline Wide shape:   (51, 9)
Long Shape Match:    PASSED
Wide Shape Match:    PASSED

--- SECTION 2: INDEX LEVEL COMPARISON ---
Maximum numerical index value discrepancy: 0.0000000000
HPI Index Levels:   PASSED (100% Identical Across All Cities & Quarters)

--- SECTION 3: QoQ & YoY LOG-GROWTH RATE PARITY ---
Max QoQ Growth rate difference: 0.0000004993
Max YoY Growth rate difference: 0.0000004971
Growth Metrics check: PASSED (Log rates match perfectly)

